1. Instalação de Dependências

Preparamos o ambiente instalando as bibliotecas necessárias para manipulação de dados (Pandas), exportação para Excel (Openpyxl) e integração com a API do Google Gemini.

In [ ]:
!pip install -q -U google-generativeai pandas openpyxl

2. Limpeza de Dados com Gemini 3 Flash (API)

Nesta etapa, utilizamos o modelo hospedado na nuvem. O script lê o dataset de vendas da cafeteria e envia cada linha para a IA analisar, corrigir erros de digitação e padronizar formatos de data e preço. O resultado é salvo em um arquivo .xlsx no diretório local.

In [ ]:
import pandas as pd
import google.generativeai as genai

GOOGLE_API_KEY = "SUA_API_KEY_AQUI"
genai.configure(api_key=GOOGLE_API_KEY)
gemini_model = genai.GenerativeModel('gemini-3-flash')

def clean_with_provider(row):
    prompt = f"Atue como um analista de qualidade de dados. Corrija inconsistências e padronize os campos desta linha: {row.to_dict()}. Retorne apenas o resultado corrigido em texto simples."
    try:
        response = gemini_model.generate_content(prompt)
        return response.text.replace("```json", "").replace("```", "").strip()
    except Exception as e:
        return f"Erro API: {e}"

caminho = "/content/dirty_cafe_sales.csv"

try:
    df = pd.read_csv(caminho, encoding='latin1')
    df_teste = df.head(10).copy()

    print("Iniciando limpeza via API...")
    df_teste['limpeza_api'] = df_teste.apply(clean_with_provider, axis=1)

    nome_arquivo = "/content/cafe_higienizado_api.xlsx"
    df_teste.to_excel(nome_arquivo, index=False)

    print(f"Processo finalizado. Arquivo disponível em: {nome_arquivo}")
    print(df_teste.head())

except FileNotFoundError:
    print(f"Erro: Arquivo '{caminho}' não encontrado no Runtime.")

3. Método Self-Hosted: Ollama

A hospedagem própria (Self-Hosted) permite rodar a IA localmente. Instalamos o servidor Ollama para gerenciar o modelo dentro do nosso próprio ambiente, garantindo que os dados nunca saiam da nossa infraestrutura.

In [1]:
!apt-get install zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q -U requests pandas openpyxl

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpci3 pci.ids
The following NEW packages will be installed:
  libpci3 pci.ids pciutils zstd
0 upgraded, 4 newly installed, 0 to remove and 42 not upgraded.
Need to get 946 kB of archives.
After this operation, 3,276 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pci.ids all 0.0~2022.01.22-1ubuntu0.1 [251 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpci3 amd64 1:3.7.0-6 [28.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 pciutils amd64 1:3.7.0-6 [63.6 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 946 kB in 1s (1,126 kB/s)
Selecting previously unselected package pci.ids.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../pci.ids_0.0~2022.01.2

4. Carregamento do Modelo Gemma 4

Realizamos o "pull" do modelo Gemma 4. Nesta etapa, o modelo é baixado e preparado para utilizar a GPU (placa de vídeo) disponível, otimizando o tempo de resposta para grandes volumes de dados.

In [7]:
import subprocess
import time

subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull gemma4:e4b-it-q4_K_M

5. Processamento e Exportação

Aplicamos a lógica de auditoria utilizando o modelo local. Após a higienização dos dados da cafeteria, o resultado é consolidado e salvo em formato Excel (.xlsx) diretamente no armazenamento do sistema para posterior download.

In [ ]:
import pandas as pd
import subprocess
import time
import requests
import json
import os

def clean_with_ollama(row):
    url = "http://localhost:11434/api/generate"

    prompt = (
        f"Atue como um auditor de dados. Corrija inconsistências nesta linha: {row.to_dict()}.\n"
        f"Seja conciso e retorne apenas o dado corrigido em texto."
    )

    payload = {
        "model": "gemma4:e4b-it-q4_K_M",
        "prompt": prompt,
        "stream": False,
    }

    try:
        response = requests.post(url, json=payload)
        if response.status_code == 200:
            return response.json()['response'].strip()
        else:
            return f"Erro Ollama ({response.status_code})"
    except Exception as e:
        return f"Erro de conexão: {str(e)}"

caminho = "/content/dirty_cafe_sales.csv"

try:
    df = pd.read_csv(caminho, encoding='latin1')
    df_teste = df.head(10).copy()
    df_teste['dados_limpos'] = df_teste.apply(clean_with_ollama, axis=1)
    df_teste.to_excel("/content/cafe_higienizado_gemma4.xlsx", index=False)
    print(df_teste.head())
except FileNotFoundError:
    print(f"Erro: O arquivo '{caminho_arquivo}' não foi encontrado.")